In [2]:
from pathlib import Path
import re
import pandas as pd


# =========================
# Locate solution folders
# =========================
def find_solution_dirs():
    current = Path.cwd().resolve()

    for parent in [current] + list(current.parents):
        vanilla = [p for p in parent.rglob("vanilla_subset*") if p.is_dir()]
        sfc = [p for p in parent.rglob("sfc_subset*") if p.is_dir()]

        if vanilla and sfc:
            print("Found solution folders:")
            print("Vanilla:", vanilla[0])
            print("SFC    :", sfc[0])
            return [vanilla[0], sfc[0]]

    raise RuntimeError("Could not find solution folders automatically.")


SOLUTION_DIRS = find_solution_dirs()

OUTPUT_DIR = Path("../../tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Parsing helpers
# =========================
def read_solution_file(path: Path) -> dict:
    data = {
        "file": str(path),
        "method": path.parent.name.replace("_subset", ""),
    }

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line == "solution":
                break

            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue

            key, value = parts

            if key in {"run", "seed", "iteration_limit", "destroy_percent"}:
                data[key] = int(value)
            elif key in {
                "runtime",
                "objective",
                "total_bin_cost",
                "total_excess",
                "total_conflicts",
            }:
                data[key] = float(value)
            else:
                data[key] = value

    data["instance_name"] = Path(data["instance_file"]).stem
    return data


def add_instance_metadata(row: dict) -> dict:
    name = row["instance_name"]

    set1_match = re.match(r"Correia_Random_(\d+)_(\d+)_(\d+)_(\d+)$", name)
    if set1_match:
        x, y, z, t = map(int, set1_match.groups())
        row.update({
            "set": "set1",
            "x": x,
            "y": y,
            "z": z,
            "t": t,
        })
        return row

    set2_match = re.match(r"HS_Random_(\d+)_(\d+)_(\d+)$", name)
    if set2_match:
        x, y, z = map(int, set2_match.groups())
        row.update({
            "set": "set2",
            "x": x,
            "y": y,
            "z": z,
        })
        return row

    row["set"] = "unknown"
    return row


def build_summary_table(df: pd.DataFrame, set_name: str) -> pd.DataFrame:
    subset = df[df["set"] == set_name].copy()

    return (
        subset
        .groupby(["method", "config", "instance_name"], as_index=False)
        .agg(
            best_obj=("objective", "min"),
            avg_obj=("objective", "mean"),
            mean_runtime=("runtime", "mean"),
            runs=("objective", "count"),
        )
        .sort_values(["config", "instance_name", "method"])
    )


# =========================
# Read solution files
# =========================
rows = []

for folder in SOLUTION_DIRS:
    for path in folder.glob("*.txt"):
        row = read_solution_file(path)
        row = add_instance_metadata(row)
        rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError("No solution files found.")


# =========================
# Generate only the two tables
# =========================
table1 = build_summary_table(df, "set1")
table2 = build_summary_table(df, "set2")

table1_path = OUTPUT_DIR / "table1_set1.csv"
table2_path = OUTPUT_DIR / "table2_set2.csv"

table1.to_csv(table1_path, index=False)
table2.to_csv(table2_path, index=False)

print("Created:")
print(table1_path)
print(table2_path)

Found solution folders:
Vanilla: /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/vanilla_subset
SFC    : /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/lnsa_solutions/sfc_subset
Created:
../../tables/table1_set1.csv
../../tables/table2_set2.csv
